In [0]:

# Create dropdown widget with file options
dbutils.widgets.dropdown(
  name="file_name",
  defaultValue="amit_donations.txt",
  choices=["amit_donations.txt", "mark_donations.txt"],
  label="Select File to Process"
)



In [0]:
# Get selected file from widget
selected_file = dbutils.widgets.get("file_name")

# Define paths
volume_path = f"/Volumes/amit/default/donations/{selected_file}"
bronze_table = "amit.default.donations_bronze"
prisoner_name = selected_file.split('_')[0]
print(f"Selected file: {selected_file}")
print(f"Source path: {volume_path}")
print(f"Target table: {bronze_table}")



In [0]:
from pyspark.sql.functions import current_timestamp, lit, row_number, col, when, concat_ws
from pyspark.sql.window import Window

# Read the text file
df = spark.read.text(volume_path)

# Filter out empty lines
df_clean = df.filter("value IS NOT NULL AND TRIM(value) != ''")

# Add line numbers
window_spec = Window.orderBy(lit(1))
df_numbered = df_clean.withColumn("line_num", row_number().over(window_spec))

# Identify donation start lines (lines that are just numbers)
df_with_donation_id = df_numbered.withColumn(
  "is_donation_start",
  col("value").rlike("^[0-9]+$")
)

# Create a temporary view for SQL processing
df_with_donation_id.createOrReplaceTempView("numbered_lines")

# Use SQL to group lines into donations
# Key insight: last line of each donation is the amount (contains ₪)
donations_df = spark.sql("""
  WITH donation_starts AS (
    SELECT 
      line_num,
      value as donation_id
    FROM numbered_lines
    WHERE is_donation_start = true
  ),
  lines_with_donation AS (
    SELECT 
      nl.line_num,
      nl.value,
      ds.donation_id,
      nl.line_num - ds.line_num as offset_in_donation
    FROM numbered_lines nl
    LEFT JOIN donation_starts ds 
      ON nl.line_num >= ds.line_num
    QUALIFY ROW_NUMBER() OVER (PARTITION BY nl.line_num ORDER BY ds.line_num DESC) = 1
  ),
  donation_structure AS (
    SELECT
      donation_id,
      MAX(CASE WHEN offset_in_donation = 1 THEN value END) as donor_name,
      MAX(CASE WHEN offset_in_donation = 2 THEN value END) as time_text,
      MAX(CASE WHEN value LIKE '%₪%' THEN value END) as amount_text,
      CONCAT_WS('\\n', 
        COLLECT_LIST(
          CASE 
            WHEN offset_in_donation > 2 AND value NOT LIKE '%₪%' 
            THEN value 
          END
        )
      ) as comment_text
    FROM lines_with_donation
    WHERE donation_id IS NOT NULL
    GROUP BY donation_id
  )
  SELECT
    donation_id,
    donor_name,
    time_text,
    amount_text,
    CASE WHEN comment_text = '' THEN NULL ELSE comment_text END as comment_text
  FROM donation_structure
""")

# Add metadata columns
df_with_metadata = donations_df \
  .withColumn("source_file", lit(selected_file)) \
  .withColumn("prisoner", lit(prisoner_name)) \
  .withColumn("ingestion_timestamp", current_timestamp())

# Write to bronze table (append mode)
df_with_metadata.write \
  .mode("append") \
  .saveAsTable(bronze_table)

print(f"✓ Successfully loaded {donations_df.count()} donations from {selected_file} to {bronze_table}")
print(f"  Prisoner: {prisoner_name}")

In [0]:
# Display the latest records from bronze table
display(
  spark.table(bronze_table)
    .filter(f"source_file = '{selected_file}'")
    .orderBy("ingestion_timestamp", ascending=False)
    .limit(20)
)

In [0]:
# Read the text file as-is (no transformations)
df = spark.read.text(volume_path)

# Extract prisoner name from filename and add metadata columns
from pyspark.sql.functions import current_timestamp, lit

# Extract prisoner name from filename (e.g., "amit_donations.txt" -> "amit")
prisoner_name = selected_file.split('_')[0]

df_with_metadata = df \
  .withColumn("source_file", lit(selected_file)) \
  .withColumn("prisoner", lit(prisoner_name)) \
  .withColumn("ingestion_timestamp", current_timestamp())

# Write to bronze table (append mode to accumulate data from multiple runs)
df_with_metadata.write \
  .mode("append") \
  .saveAsTable(bronze_table)

print(f"✓ Successfully loaded {df.count()} rows from {selected_file} to {bronze_table}")
print(f"  Prisoner: {prisoner_name}")


In [0]:
Cell 4: Verify the Load
# Display the latest records from bronze table
display(
  spark.table(bronze_table)
    .filter(f"source_file = '{selected_file}'")
    .orderBy("ingestion_timestamp", ascending=False)
    .limit(20)
)

Usage Instructions
Run Cell 1 to create the dropdown widget at the top of the notebook
Select a file from the dropdown (e.g., amit_donations.txt)
Run Cell 2 to configure paths based on your selection
Run Cell 3 to load the raw data into the bronze table
Run Cell 4 to verify the data was loaded correctly
Schema
The bronze table will have this schema:
Column
Type
Description
value
string
Raw line from the text file (unchanged)
source_file
string
Name of the source file
prisoner
string
Prisoner name extracted from filename (first part before underscore)
ingestion_timestamp
timestamp
When the data was loaded
Notes
No transformations: Data is loaded exactly as-is from the source file
Append mode: Each run adds new rows (doesn't overwrite)
Metadata tracking: source_file, prisoner, and ingestion_timestamp help track data lineage
Prisoner extraction: Automatically extracts prisoner name from filename pattern {prisoner}_donations.txt
Idempotency: To avoid duplicates, you can change mode to overwrite or add deduplication logic in a later silver layer
Extending the Pattern
Add More Files
To add more file options, update Cell 1:
dbutils.widgets.dropdown(
  name="file_name",
  defaultValue="amit_donations.txt",
  choices=[
    "amit_donations.txt", 
    "mark_donations.txt",
    "sarah_donations.txt",
    "david_donations.txt"
  ],
  label="Select File to Process"
)
Dynamic File Discovery
To automatically discover all files in the volume:
# Cell 1 alternative: Auto-discover files
volume_dir = "/Volumes/amit/default/donations/"
files = [f for f in dbutils.fs.ls(volume_dir) if f.name.endswith('.txt')]
file_names = [f.name for f in files]

dbutils.widgets.dropdown(
  name="file_name",
  defaultValue=file_names[0] if file_names else "",
  choices=file_names,
  label="Select File to Process"
)
Change to Overwrite Mode
If you want each run to replace the data for that file (instead of appending):
# In Cell 3, change the write mode
df_with_metadata.write \
  .mode("overwrite") \
  .option("replaceWhere", f"source_file = '{selected_file}'") \
  .saveAsTable(bronze_table)
Custom Prisoner Name Extraction
If your filename pattern is different, adjust the extraction logic in Cell 3:
# For pattern: "donations_amit.txt"
prisoner_name = selected_file.split('_')[1].replace('.txt', '')

# For pattern: "amit-donations.txt"
prisoner_name = selected_file.split('-')[0]

# For explicit mapping
prisoner_mapping = {
  "amit_donations.txt": "amit",
  "mark_donations.txt": "mark",
  "special_file.txt": "custom_name"
}
prisoner_name = prisoner_mapping.get(selected_file, "unknown")